# Durable model — infinite-horizon results (all channels)

Load steady states and sequence-space Jacobians produced by:

- `DurableModel_Taste_Inf.ipynb` (EV taste shocks)
- `DurableModel_Search_Inf.ipynb` (directed search)
- `DurableModel_Price_Inf.ipynb` (price dispersion)
- `DurableModel_Combined_varP_Inf.ipynb` (search + price, vary $\sigma_p$)
- `DurableModel_Combined_varS_Inf.ipynb` (search + price, vary $S$)

Each channel contributes three pickles under `Output/`:

- `DurableModel_{Taste,Search,Price,Combined_varP,Combined_varS}_Inf_SS.pkl`
- `DurableModel_{Taste,Search,Price,Combined_varP,Combined_varS}_Inf_Jacobians_Rfree.pkl`
- `DurableModel_{Taste,Search,Price,Combined_varP,Combined_varS}_Inf_Jacobians_labor.pkl`

Run those notebooks first if the files are missing or stale.

In [1]:
import pickle
from pathlib import Path

import pandas as pd
from IPython.display import display

OUT_DIR = Path("../Output")
TABLE_DIR = Path("../Tables")

CHANNELS = ("Taste", "Search", "Price", "Combined_varP", "Combined_varS")
CHANNEL_LABELS = {
    "taste": r"$\sigma_\varepsilon$",
    "search": r"$S$",
    "price": r"$\sigma_p$",
    "combined_varp": r"$\sigma_p$ ($S=0.2$)",
    "combined_vars": r"$S$ ($\sigma_p=0.1$)",
}
SS_KEYS = ["C", "D", "DEX", "TEX", "A", "ADJ"]
JAC_KEYS = ["C", "DEX", "TEX", "D", "A", "ADJ"]
TABLE_KEYS = SS_KEYS  # steady-state aggregates only (no labor MPCs)
SS_CHANNELS_SHORT = ("taste", "combined_varp", "combined_vars")
LINE_STYLES = [
    ("k", "-"),
    ("r", "-"),
    ("k", "--"),
    ("k", ":"),
]

In [2]:
def _load_pkl(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path.resolve()}. Run the corresponding *_Inf.ipynb first."
        )
    with open(path, "rb") as f:
        return pickle.load(f)


def load_channel(channel, out_dir=OUT_DIR):
    """Load SS + Rfree/labor Jacobian pickles for one channel name."""
    stem = f"DurableModel_{channel}_Inf"
    ss = _load_pkl(out_dir / f"{stem}_SS.pkl")
    jac_Rfree = _load_pkl(out_dir / f"{stem}_Jacobians_Rfree.pkl")
    jac_labor = _load_pkl(out_dir / f"{stem}_Jacobians_labor.pkl")
    return {
        "channel": channel.lower(),
        "ss": ss,
        "jacobians_Rfree": jac_Rfree,
        "jacobians_labor": jac_labor,
    }


data = {ch.lower(): load_channel(ch) for ch in CHANNELS}

for ch, bundle in data.items():
    ss = bundle["ss"]
    jr = bundle["jacobians_Rfree"]
    jl = bundle["jacobians_labor"]
    print(
        f"{ch}: param={ss['param_name']}, "
        f"values={ss['param_values']}, "
        f"T_Rfree={jr['T']}, T_labor={jl['T']}, "
        f"shk=({jr['shk_param']}, {jl['shk_param']})"
    )

taste: param=sigma_eps, values=[0.05, 0.1, 0.2], T_Rfree=50, T_labor=50, shk=(Rfree, labor)
search: param=SearchCost, values=[0.15, 0.2, 0.25], T_Rfree=50, T_labor=50, shk=(Rfree, labor)
price: param=PriceShkStd, values=[0.05, 0.1, 0.15], T_Rfree=50, T_labor=50, shk=(Rfree, labor)
combined_varp: param=PriceShkStd, values=[0.08, 0.1, 0.12], T_Rfree=50, T_labor=50, shk=(Rfree, labor)
combined_vars: param=SearchCost, values=[0.15, 0.2, 0.25], T_Rfree=50, T_labor=50, shk=(Rfree, labor)


## Steady-state aggregates

Combined comparison across taste / search / price / combined (vary $\sigma_p$ and vary $S$), same grouped
layout as `Tables/smoothing_euler.tex` (multirow channel label + value column).

Tabular body only is written to `Output/DurableModel_Inf_SS.tex`.


In [3]:
def _fmt_ss(x, nd=2):
    """Format a steady-state level for LaTeX math mode."""
    return f"{float(x):.{nd}f}"


def _fmt_nd(key, nd=2, adj_nd=1):
    """Decimals for table cells; ADJ is already in percent."""
    return adj_nd if key == "ADJ" else nd


def build_ss_comparison_rows(data, ss_keys=SS_KEYS):
    """
    Flat rows for display / export.

    Each row: channel TeX label, parameter value, and SS aggregates.
    ADJ is stored as share of adjusters in percent (×100).

    Combined_varP holds $S=0.2$ and varies $\sigma_p$; Combined_varS holds
    $\sigma_p=0.1$ and varies $S$.
    """
    channel_order = tuple((k, CHANNEL_LABELS[k]) for k in CHANNEL_LABELS)
    rows = []
    for ch_key, ch_tex in channel_order:
        ss = data[ch_key]["ss"]
        params = list(ss["param_values"])
        for p in params:
            vals = ss["steady_states"][p]
            row = {
                "channel": ch_tex,
                "value": p,
                **{k: float(vals[k]) for k in ss_keys},
            }
            if "ADJ" in row:
                row["ADJ"] *= 100.0
            rows.append(row)
    return rows


def ss_comparison_dataframe(rows, table_keys=TABLE_KEYS):
    df = pd.DataFrame(rows)
    df = df.set_index(["channel", "value"])[table_keys]
    return df


def export_ss_comparison_table(rows, path=None, ss_keys=SS_KEYS, nd=2, adj_nd=1):
    """
    Write a tabular body matching ``Tables/smoothing_euler.tex`` layout.

    Columns: channel | value | SS (C, D, DEX, TEX, A, ADJ %).
    Caption / label stay outside this file.
    """
    path = Path(path) if path is not None else OUT_DIR / "DurableModel_Inf_SS.tex"
    path.parent.mkdir(parents=True, exist_ok=True)

    table_keys = list(ss_keys)
    n_ss = len(ss_keys)
    colspec = "ll" + "c" * n_ss

    ss_header_names = {
        "C": r"$C$",
        "D": r"$D$",
        "DEX": r"$\mathrm{DEX}$",
        "TEX": r"$\mathrm{TEX}$",
        "A": r"$A$",
        "ADJ": r"$\mathrm{ADJ}$ (\%)",
    }

    top = " &  & " + rf"\multicolumn{{{n_ss}}}{{c}}{{Steady state}} \\"
    clines = rf"\cline{{3-{2 + n_ss}}}"
    sub = (
        " &  & "
        + " & ".join(ss_header_names[k] for k in ss_keys)
        + r" \\"
    )

    groups = []
    for row in rows:
        if not groups or groups[-1][0] != row["channel"]:
            groups.append((row["channel"], []))
        groups[-1][1].append(row)

    lines = [
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\hline",
        top,
        clines,
        sub,
        r"\hline",
    ]
    for g, (ch, items) in enumerate(groups):
        if g > 0:
            lines.append(r"\addlinespace")
        n = len(items)
        for i, row in enumerate(items):
            left = rf"\multirow{{{n}}}{{*}}{{{ch}}}" if i == 0 else ""
            nums = " & ".join(
                f"${_fmt_ss(row[k], nd=_fmt_nd(k, nd=nd, adj_nd=adj_nd))}$"
                for k in table_keys
            )
            val = row["value"]
            val_tex = f"${val}$" if isinstance(val, (int, float)) else str(val)
            lines.append(rf"{left} & {val_tex} & {nums} \\")
    lines += [r"\hline", r"\end{tabular}", ""]
    path.write_text("\n".join(lines), encoding="utf-8")
    print("Wrote", path.resolve())
    return path


ss_rows = build_ss_comparison_rows(data)
ss_df = ss_comparison_dataframe(ss_rows)
print("Steady-state aggregates by channel")
_fmt = {k: "{:.2f}" for k in TABLE_KEYS}
_fmt["ADJ"] = "{:.1f}"
display(ss_df.style.format(_fmt))

ss_tex_path = export_ss_comparison_table(ss_rows)
print(ss_tex_path.read_text(encoding="utf-8"))

# Short table for slides: taste + combined_varP + combined_varS
ss_rows_short = [r for r in ss_rows if r["channel"] in {CHANNEL_LABELS[k] for k in SS_CHANNELS_SHORT}]
ss_df_short = ss_comparison_dataframe(ss_rows_short)
print("Short steady-state table (taste, combined_varP, combined_varS)")
display(ss_df_short.style.format(_fmt))
ss_tex_path_short = export_ss_comparison_table(
    ss_rows_short, path=OUT_DIR / "DurableModel_Inf_SS_short.tex"
)
print(ss_tex_path_short.read_text(encoding="utf-8"))


Steady-state aggregates by channel


Wrote C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Output\DurableModel_Inf_SS.tex
\begin{tabular}{llcccccc}
\hline
 &  & \multicolumn{6}{c}{Steady state} \\
\cline{3-8}
 &  & $C$ & $D$ & $\mathrm{DEX}$ & $\mathrm{TEX}$ & $A$ & $\mathrm{ADJ}$ (\%) \\
\hline
\multirow{3}{*}{$\sigma_\varepsilon$} & $0.05$ & $0.82$ & $3.38$ & $0.17$ & $0.99$ & $1.74$ & $9.8$ \\
 & $0.1$ & $0.80$ & $3.31$ & $0.17$ & $0.97$ & $1.25$ & $13.5$ \\
 & $0.2$ & $0.79$ & $2.97$ & $0.15$ & $0.94$ & $0.66$ & $25.0$ \\
\addlinespace
\multirow{3}{*}{$S$} & $0.15$ & $0.85$ & $3.21$ & $0.16$ & $1.01$ & $2.51$ & $6.2$ \\
 & $0.2$ & $0.84$ & $3.25$ & $0.16$ & $1.00$ & $2.46$ & $6.6$ \\
 & $0.25$ & $0.84$ & $3.24$ & $0.16$ & $1.00$ & $2.45$ & $6.6$ \\
\addlinespace
\multirow{3}{*}{$\sigma_p$} & $0.05$ & $0.81$ & $4.07$ & $0.20$ & $1.01$ & $1.30$ & $12.4$ \\
 & $0.1$ & $0.81$ & $5.08$ & $0.27$ & $1.08$ & $0.76$ & $20.0$ \\
 & $0.15$ & $0.73$ & $5.47$ & $0.47$ & $1.20$ & $1.06$ & $20.0$ \

Wrote C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Output\DurableModel_Inf_SS_short.tex
\begin{tabular}{llcccccc}
\hline
 &  & \multicolumn{6}{c}{Steady state} \\
\cline{3-8}
 &  & $C$ & $D$ & $\mathrm{DEX}$ & $\mathrm{TEX}$ & $A$ & $\mathrm{ADJ}$ (\%) \\
\hline
\multirow{3}{*}{$\sigma_\varepsilon$} & $0.05$ & $0.82$ & $3.38$ & $0.17$ & $0.99$ & $1.74$ & $9.8$ \\
 & $0.1$ & $0.80$ & $3.31$ & $0.17$ & $0.97$ & $1.25$ & $13.5$ \\
 & $0.2$ & $0.79$ & $2.97$ & $0.15$ & $0.94$ & $0.66$ & $25.0$ \\
\addlinespace
\multirow{3}{*}{$\sigma_p$ ($S=0.2$)} & $0.08$ & $0.83$ & $4.35$ & $0.22$ & $1.05$ & $1.60$ & $9.8$ \\
 & $0.1$ & $0.83$ & $4.87$ & $0.24$ & $1.07$ & $1.35$ & $11.6$ \\
 & $0.12$ & $0.83$ & $5.64$ & $0.28$ & $1.11$ & $1.18$ & $13.8$ \\
\addlinespace
\multirow{3}{*}{$S$ ($\sigma_p=0.1$)} & $0.15$ & $0.83$ & $4.98$ & $0.25$ & $1.08$ & $1.28$ & $12.4$ \\
 & $0.2$ & $0.83$ & $4.87$ & $0.24$ & $1.07$ & $1.35$ & $11.6$ \\
 & $0.25$ & $0.83$ & $4.83$ & 

## Kinks and Euler residuals (infinite horizon)

Same diagnostic table as `Figures_Durable.ipynb` (`Tables/smoothing_euler.tex`),
for the Inf models across taste / search / price / combined (varP and varS). Period `t = 0`
is the stationary policy: kink counts live on `solution[0]` after solve; Euler
residuals are filled by `compute_euler_errors()` in each channel notebook and
saved under `kink_euler` in `*_Inf_SS.pkl`.

Tabular body only is written to `Tables/smoothing_euler_Inf.tex`.


In [4]:
def _fmt_int(x):
    if x is None or (isinstance(x, float) and (x != x)):
        return "---"
    return f"{int(x)}"


def _fmt_share(x):
    if x is None or (isinstance(x, float) and (x != x)):
        return "---"
    if abs(float(x)) < 1e-15:
        return "0"
    return f"{float(x):.3f}"


def _fmt_ee(x):
    if x is None or (isinstance(x, float) and (x != x)):
        return "---"
    return f"{float(x):.2f}"


def build_kink_euler_rows(data):
    """Load period-0 kink / Euler rows saved in each channel SS pickle."""
    channel_order = tuple(CHANNEL_LABELS.keys())
    rows = []
    missing = []
    for ch_key in channel_order:
        ss = data[ch_key]["ss"]
        ke = ss.get("kink_euler")
        if not ke:
            missing.append(ch_key)
            continue
        for p, lab in zip(ss["param_values"], ss["labels"]):
            row = dict(ke[p])
            row.setdefault("Specification", lab)
            row["channel"] = CHANNEL_LABELS[ch_key]
            row["value"] = p
            rows.append(row)
    if missing:
        raise KeyError(
            "Missing kink_euler in SS pickles for: "
            + ", ".join(missing)
            + ". Re-run DurableModel_{Taste,Search,Price,Combined_varP,Combined_varS}_Inf.ipynb "
            "(they now store solution[0] kink counts and Euler residuals)."
        )
    return rows, int(data["taste"]["ss"].get("kink_euler_t", 0))


def kink_euler_dataframe(rows):
    cols = [
        "Specification",
        "Decr. seg.",
        "Keep share",
        "FUES rem.",
        "Adj share",
        "EE_c keep mean",
        "EE_c keep p95",
        "EE_c adj mean",
        "EE_c adj p95",
        "EE_d adj mean",
        "EE_d adj p95",
    ]
    df = pd.DataFrame(rows)[cols]
    for c in ("Decr. seg.", "FUES rem."):
        df[c] = df[c].map(_fmt_int)
    for c in ("Keep share", "Adj share"):
        df[c] = df[c].map(_fmt_share)
    for c in df.columns:
        if c.startswith("EE_"):
            df[c] = df[c].map(_fmt_ee)
    return df


def export_inf_euler_table(rows, path=None):
    """
    Write ``Tables/smoothing_euler_Inf.tex`` with the same grouped layout as
    ``Tables/smoothing_euler.tex`` (Inf sweeps only; no hard-max row).
    """
    path = Path(path) if path is not None else TABLE_DIR / "smoothing_euler_Inf.tex"
    path.parent.mkdir(parents=True, exist_ok=True)

    def _ee_tex(x):
        s = _fmt_ee(x)
        return s if s == "---" else f"${s}$"

    def _nums(row):
        return " & ".join(
            [
                _fmt_int(row["Decr. seg."]),
                _fmt_int(row["FUES rem."]),
                _ee_tex(row["EE_c keep mean"]),
                _ee_tex(row["EE_c keep p95"]),
                _ee_tex(row["EE_d adj mean"]),
                _ee_tex(row["EE_d adj p95"]),
            ]
        )

    groups = []
    for row in rows:
        ch, val = row["channel"], row["value"]
        if not groups or groups[-1][0] != ch:
            groups.append((ch, []))
        groups[-1][1].append((val, row))

    lines = [
        r"\begin{tabular}{llcccccc}",
        r"\hline",
        r" &  & \multicolumn{2}{c}{Kinks} & \multicolumn{2}{c}{$\mathrm{EE}_c^{\mathrm{keep}}$} & \multicolumn{2}{c}{$\mathrm{EE}_d^{\mathrm{adj}}$} \\",
        r"\cline{3-4}\cline{5-6}\cline{7-8}",
        r" &  & Decr.\ seg. & FUES rem. & Mean & P95 & Mean & P95 \\",
        r"\hline",
    ]
    for g, (ch, items) in enumerate(groups):
        if g > 0:
            lines.append(r"\addlinespace")
        n = len(items)
        for i, (val, row) in enumerate(items):
            left = rf"\multirow{{{n}}}{{*}}{{{ch}}}" if i == 0 else ""
            lines.append(rf"{left} & ${val}$ & {_nums(row)} \\")
    lines += [r"\hline", r"\end{tabular}", ""]
    path.write_text("\n".join(lines), encoding="utf-8")
    print("Wrote", path.resolve())
    return path


diag_rows, t_ee = build_kink_euler_rows(data)
print(f"Period t = {t_ee} (infinite-horizon stationary policy)\n")
display(kink_euler_dataframe(diag_rows))
ee_tex_path = export_inf_euler_table(diag_rows)
print(ee_tex_path.read_text(encoding="utf-8"))


Period t = 0 (infinite-horizon stationary policy)



,Specification,Decr. seg.,Keep share,FUES rem.,Adj share,EE_c keep mean,EE_c keep p95,EE_c adj mean,EE_c adj p95,EE_d adj mean,EE_d adj p95
0,$\sigma_{\varepsilon} = 0.05$,0,0,0,0,-3.92,-0.81,-2.48,-0.02,-2.29,0.67
1,$\sigma_{\varepsilon} = 0.1$,0,0,0,0,-5.08,-4.09,-2.48,-0.13,-1.49,1.35
2,$\sigma_{\varepsilon} = 0.2$,0,0,0,0,-5.29,-4.17,-2.81,-0.33,-2.17,0.66
3,$S = 0.15$,1,0.005,0,0,-3.59,-2.23,-2.39,0.01,-1.24,1.59
4,$S = 0.2$,0,0,0,0,-3.89,-2.64,-2.32,0.03,-2.24,0.50
5,$S = 0.25$,0,0,0,0,-4.80,-3.60,-2.43,0.03,-2.05,0.53
6,$\sigma_{p} = 0.05$,0,0,102,0.436,-4.76,-3.31,-2.86,-0.15,-1.89,1.24
7,$\sigma_{p} = 0.1$,0,0,59,0.273,-5.01,-3.83,-3.62,-0.69,-2.73,0.10
8,$\sigma_{p} = 0.15$,1,0.005,22,0.105,-4.92,-3.87,-3.96,-2.73,-3.20,-1.00
9,"$S = 0.2$, $\sigma_{p} = 0.08$",0,0,32,0.111,-5.09,-4.25,-3.12,-0.19,-2.31,0.98


Wrote C:\Users\amonninger\OneDrive - Texas A&M University\GitHub\AMonninger\DCEGM_Search\Tables\smoothing_euler_Inf.tex
\begin{tabular}{llcccccc}
\hline
 &  & \multicolumn{2}{c}{Kinks} & \multicolumn{2}{c}{$\mathrm{EE}_c^{\mathrm{keep}}$} & \multicolumn{2}{c}{$\mathrm{EE}_d^{\mathrm{adj}}$} \\
\cline{3-4}\cline{5-6}\cline{7-8}
 &  & Decr.\ seg. & FUES rem. & Mean & P95 & Mean & P95 \\
\hline
\multirow{3}{*}{$\sigma_\varepsilon$} & $0.05$ & 0 & 0 & $-3.92$ & $-0.81$ & $-2.29$ & $0.67$ \\
 & $0.1$ & 0 & 0 & $-5.08$ & $-4.09$ & $-1.49$ & $1.35$ \\
 & $0.2$ & 0 & 0 & $-5.29$ & $-4.17$ & $-2.17$ & $0.66$ \\
\addlinespace
\multirow{3}{*}{$S$} & $0.15$ & 1 & 0 & $-3.59$ & $-2.23$ & $-1.24$ & $1.59$ \\
 & $0.2$ & 0 & 0 & $-3.89$ & $-2.64$ & $-2.24$ & $0.50$ \\
 & $0.25$ & 0 & 0 & $-4.80$ & $-3.60$ & $-2.05$ & $0.53$ \\
\addlinespace
\multirow{3}{*}{$\sigma_p$} & $0.05$ & 0 & 102 & $-4.76$ & $-3.31$ & $-1.89$ & $1.24$ \\
 & $0.1$ & 0 & 59 & $-5.01$ & $-3.83$ & $-2.73$ & $0.10$ \\
 & $0.15$ & 1 

## Jacobian access

Payload layout (same for `jacobians_Rfree` and `jacobians_labor`):

```python
data["taste"]["jacobians_Rfree"]["jacobians"][param_value]["C"]  # T x T array
```

Channel keys: `taste`, `search`, `price`, `combined_varp`, `combined_vars`.

Keys in each Jacobian dict: `C`, `DEX`, `TEX`, `D`, `A`, `ADJ`.

In [5]:
# Example: shapes for the first parameter value of each channel / shock
for ch, bundle in data.items():
    for shock_key in ("jacobians_Rfree", "jacobians_labor"):
        jac = bundle[shock_key]
        p0 = jac["param_values"][0]
        shapes = {k: jac["jacobians"][p0][k].shape for k in JAC_KEYS}
        print(f"{ch} | {jac['shk_param']} | param={p0}: {shapes}")

taste | Rfree | param=0.05: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
taste | labor | param=0.05: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
search | Rfree | param=0.15: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
search | labor | param=0.15: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
price | Rfree | param=0.05: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
price | labor | param=0.05: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
combined_varp | Rfree | param=0.08: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50), 'ADJ': (50, 50)}
combined_varp | labor | param=0.08: {'C': (50, 50), 'DEX': (50, 50), 'TEX': (50, 50), 'D': (50, 50), 'A': (50, 50),